# Hooks Deep Dive: Events & Architecture

A **hook** is an async function that you register for a named event. When the event fires, the SDK calls your function, waits for it, and acts on what it returns.

In this notebook you will see the whole hook architecture: the ten Python hook events, how a registration is built, what a callback receives and returns, and how several hooks combine when they run together. Each idea is proved with a small demo that produces visible evidence.

| Demo | What it proves | Evidence you will see |
|---|---|---|
| 1. Protect a folder | A `PreToolUse` hook can block a tool call | One file that exists and one that does not |
| 2. Event tracer | One hook can listen to several events, and `tool_use_id` links a before and after pair | A printed trace |
| 3. Matchers | A matcher filters by tool name only | A counter |
| 4. Parallel hooks | When hooks disagree, deny wins | A file that does not exist |

**A note on live output.** Claude is an agent, so its wording, and sometimes its steps, vary from run to run. That is normal. Each demo ends with something you can check objectively, such as a file on disk or a counter. Judge the demo by that evidence, not by the exact words Claude replies with.

## Setup 1: Install the Claude Agent SDK

This cell installs the `claude-agent-sdk` package, which is the only package this notebook needs.

| Part | Meaning |
|---|---|
| `!` | Tells Colab to run the line in the shell instead of as Python |
| `==0.2.163` | Pins one exact version so every example behaves the way it does in the videos |
| `-q` | Keeps the install output quiet |

The comment at the top of the cell explains the pin. Pinning gives you reproducible results. If you want the newest release, run `pip install -U claude-agent-sdk` instead, or change the version number to one you prefer. Newer versions may behave differently from the videos.

If the package is already present in your current Colab session, pip notices and finishes almost immediately.

In [1]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.163 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.6 MB/s eta 0:00:00


## Setup 2: Add your API key

The agent needs an Anthropic API key. In Colab we store it in **Secrets** so that the key never appears in the notebook.

**If you do not have a key yet,** create one in the Claude Console at https://platform.claude.com/ under **API Keys**.

**Add the key to Colab Secrets:**

1. Click the **key icon** in the left sidebar.
2. Click **Add new secret**.
3. Set the name to exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value.
5. Switch **Notebook access** on.

The cell below reads the secret with `userdata.get(...)` and writes it into the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it.

*Running locally instead of in Colab?* Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before starting Python.

In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Setup 3: Choose the model

We keep the model name in one variable, `MODEL_NAME`. Every agent run in this notebook passes `model=MODEL_NAME`, so **changing this single line changes the model for the entire notebook**.

We use `claude-haiku-4-5`, a fast and economical model that is more than capable for these demos. Model names change as new releases arrive, so the comment in the cell links to the models page where you can find the latest options.

In [3]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Setup 4: Imports and the sample project

Every demo works on the same small project, which we create with code so that you start from a known state.

```
/content/hooks_demo/
    drafts/
        meeting-notes.txt     (five short lines containing three action items)
    published/
        changelog.txt         (two lines)
```

The `drafts/` folder is where work in progress lives. The `published/` folder is finished work that should not be edited by accident. Demo 1 turns that rule into a hook.

The imports at the top are the only SDK imports this notebook needs:

| Name | Purpose |
|---|---|
| `query` | Starts an agent run and returns an async stream of messages |
| `ClaudeAgentOptions` | Holds the run's settings, including `hooks`, `allowed_tools`, `model` and `cwd` |
| `HookMatcher` | Describes one hook registration: an optional matcher, the hooks to call, and an optional timeout |

In [4]:
import os
from pathlib import Path
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher

PROJECT = Path("/content/hooks_demo")
(PROJECT / "drafts").mkdir(parents=True, exist_ok=True)
(PROJECT / "published").mkdir(parents=True, exist_ok=True)

meeting_notes = """Weekly sync, 12 March
Attendees: Asha, Ravi, Meera
Action item 1: Asha sends the revised schedule by Friday.
Action item 2: Ravi updates the changelog for version 1.2.
Action item 3: Meera books the review room for Monday.
"""

changelog = """Version 1.1: fixed the login timeout.
Version 1.2: added the export button.
"""

(PROJECT / "drafts" / "meeting-notes.txt").write_text(meeting_notes)
(PROJECT / "published" / "changelog.txt").write_text(changelog)

print("Sample project created.")

Sample project created.


## Setup 5: Verify the sample project

Before any agent runs, we confirm exactly what is on disk. The first loop prints the directory tree. The second loop prints the contents of each file.

You should see two folders, two files, and three action items inside `meeting-notes.txt`. Keep this picture in mind, because every demo is judged against it.

In [5]:
for folder, _, files in sorted(os.walk(PROJECT)):
    depth = len(Path(folder).relative_to(PROJECT).parts)
    indent = "    " * depth
    print(f"{indent}{Path(folder).name}/")
    for name in sorted(files):
        print(f"{indent}    {name}")

print()
for path in sorted(PROJECT.rglob("*.txt")):
    print(f"--- {path.relative_to(PROJECT)} ---")
    print(path.read_text())

hooks_demo/
    drafts/
        meeting-notes.txt
    published/
        changelog.txt

--- drafts/meeting-notes.txt ---
Weekly sync, 12 March
Attendees: Asha, Ravi, Meera
Action item 1: Asha sends the revised schedule by Friday.
Action item 2: Ravi updates the changelog for version 1.2.
Action item 3: Meera books the review room for Monday.

--- published/changelog.txt ---
Version 1.1: fixed the login timeout.
Version 1.2: added the export button.



## Recap: the hook you already know

You have already registered a `PreToolUse` hook. The SDK calls it just before a tool runs. The hook looks at `tool_name` and `tool_input`, and it can return `permissionDecision: "deny"` to block the call, or `updatedInput` to rewrite it.

The dummy hook you used to keep other features working was also a `PreToolUse` hook. It returned `{"continue_": True}` and made no decision.

That was **one event out of ten**. The rest of this notebook gives you the whole picture. We begin with Demo 1, a hook that blocks, so that the architecture that follows has something concrete to attach to.

## Demo 1 (part 1): A hook that protects `published/`

This is the blocking pattern again, with a new scenario: files in `published/` are read-only.

Read the callback from top to bottom:

1. `input_data["tool_input"].get("file_path", "")` reads the path the tool is about to use. `.get` with a default keeps the hook safe for tools that have no `file_path`.
2. If the path contains `/published/`, the hook prints a `[BLOCKED]` line and returns an object.
3. The object puts `permissionDecision: "deny"` **inside `hookSpecificOutput`**, together with `hookEventName` and a reason. `permissionDecisionReason` is shown to Claude so it can adapt, for example by writing to `drafts/` instead.
4. Otherwise the hook returns `{}`, which changes nothing.

The signature `(input_data, tool_use_id, context)` is the same for every hook. We will look at each argument shortly.

In [6]:
async def protect_published(input_data, tool_use_id, context):
    print(input_data)
    file_path = input_data["tool_input"].get("file_path", "")
    if "/published/" in file_path:
        print(f"[BLOCKED] {input_data['tool_name']} -> {file_path}")
        return {
            "hookSpecificOutput": {
                "hookEventName": input_data["hook_event_name"],
                "permissionDecision": "deny",
                "permissionDecisionReason": "Files in published/ are read-only. Write drafts in drafts/ instead.",
            }
        }
    return {}

## Demo 1 (part 2): Run the agent with the hook registered

Remember that `query()` does not return a value. It returns an **async stream of messages** while the agent loop runs on its own. We print only the final answer, which arrives on the result message.

Three settings matter here:

| Setting | Why |
|---|---|
| `allowed_tools=["Read", "Write"]` | `Write` is pre-approved. Without our hook, both writes would succeed, so a block is provably the hook's doing. |
| `hooks={"PreToolUse": [HookMatcher(matcher="Write\|Edit", hooks=[protect_published])]}` | Registers the hook for `PreToolUse`, but only for the `Write` and `Edit` tools |
| `cwd="/content"` | Keeps tool calls inside the Colab working area |

The loop at the top deletes any files left over from an earlier run, so the result you see belongs to this run. The prompt asks for two files, one in each folder. Look for a single `[BLOCKED]` line.

In [7]:
for name in ["drafts/action-items.txt", "published/action-items.txt"]:
    target = PROJECT / name
    if target.exists():
        target.unlink()

async for message in query(
    prompt=(
        "Read /content/hooks_demo/drafts/meeting-notes.txt. "
        "Then copy its action items into two new files: "
        "/content/hooks_demo/drafts/action-items.txt and "
        "/content/hooks_demo/published/action-items.txt."
    ),
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write"],
        hooks={
            "PreToolUse": [
                HookMatcher(matcher="Write|Edit", hooks=[protect_published]),
            ],
        },
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if hasattr(message, "result"):
        print(message.result)

{'session_id': 'b2797aae-ff5e-4868-b9d8-54e1faa684ab', 'transcript_path': '/root/.claude/projects/-content/b2797aae-ff5e-4868-b9d8-54e1faa684ab.jsonl', 'cwd': '/content', 'prompt_id': '900a82d8-bf6f-4e8c-9fa8-6ab549f2dbed', 'permission_mode': 'default', 'hook_event_name': 'PreToolUse', 'tool_name': 'Write', 'tool_input': {'file_path': '/content/hooks_demo/drafts/action-items.txt', 'content': 'Action Items from Weekly Sync, 12 March\n\n1. Asha sends the revised schedule by Friday.\n2. Ravi updates the changelog for version 1.2.\n3. Meera books the review room for Monday.\n'}, 'tool_use_id': 'toolu_019Y5ZP7jyDJrBmCoGCANzCN'}
{'session_id': 'b2797aae-ff5e-4868-b9d8-54e1faa684ab', 'transcript_path': '/root/.claude/projects/-content/b2797aae-ff5e-4868-b9d8-54e1faa684ab.jsonl', 'cwd': '/content', 'prompt_id': '900a82d8-bf6f-4e8c-9fa8-6ab549f2dbed', 'permission_mode': 'default', 'hook_event_name': 'PreToolUse', 'tool_name': 'Write', 'tool_input': {'file_path': '/content/hooks_demo/published/a

## Demo 1 (part 3): Ground truth

Claude's reply will vary, so we do not trust the wording. We ask the filesystem instead.

**Expected:** `drafts/action-items.txt` exists is `True`, and `published/action-items.txt` exists is `False`. Because `Write` was in `allowed_tools`, nothing but the hook could have stopped that second write.

In [8]:
for name in ["drafts/action-items.txt", "published/action-items.txt"]:
    print(f"{name:30} exists = {(PROJECT / name).exists()}")

drafts/action-items.txt        exists = True
published/action-items.txt     exists = False


## The ten Python hook events

The Python SDK supports ten hook events. The key you use in the `hooks` dictionary is the event name, and it is **case sensitive**.

| Event | Fires | Typical use |
|---|---|---|
| `PreToolUse` | before a tool runs | block or modify the call |
| `PostToolUse` | after a tool returns | audit log, add context, replace output |
| `PostToolUseFailure` | when a tool fails | log or handle errors |
| `UserPromptSubmit` | when a prompt is submitted | inject context |
| `Stop` | when the agent stops | final summary, save state |
| `SubagentStart` | when a subagent starts | track spawning |
| `SubagentStop` | when a subagent finishes | collect results |
| `PreCompact` | before conversation compaction | archive the transcript |
| `PermissionRequest` | when a permission decision is needed | custom permission handling |
| `Notification` | agent status messages | forward status updates |

The documentation lists further events. This course uses the ones that are available in the Python SDK. This section focuses on `PreToolUse`, `PostToolUse`, `PostToolUseFailure`, `UserPromptSubmit` and `Stop`.

## Anatomy of a registration

Hooks are registered through the `hooks` option of `ClaudeAgentOptions`.

```python
hooks={
    "PreToolUse": [HookMatcher(matcher="Write|Edit", hooks=[my_hook], timeout=30)],
    "PostToolUse": [HookMatcher(hooks=[audit_hook])],
}
```

- **Keys** are event names, and they are case sensitive.
- **Values** are lists of `HookMatcher`.
- Each `HookMatcher` has an optional `matcher`, a required list of `hooks`, and an optional `timeout` in seconds.

**How `matcher` works.** For tool events, the matcher is tested against the **tool name**. It can be exact (`"Bash"`), a pipe list (`"Write|Edit"`) or a regex (`"^mcp__"`). It never matches file paths or arguments, so path checks happen inside the hook, using `tool_input`. If you omit `matcher`, the hook runs on every occurrence of the event. `Stop` ignores matchers.

**What a callback receives and returns.**

```python
async def my_hook(input_data, tool_use_id, context):
    return {}
```

`input_data` always contains `hook_event_name`, `session_id`, `transcript_path` and `cwd`, and it may contain `permission_mode`. Each event adds its own fields:

| Event | Extra fields in `input_data` |
|---|---|
| `PreToolUse` | `tool_name`, `tool_input`, `tool_use_id` |
| `PostToolUse` | the same fields, plus `tool_response` |
| `PostToolUseFailure` | `tool_name`, `tool_input`, `tool_use_id`, `error`, and sometimes `is_interrupt` |
| `UserPromptSubmit` | `prompt` |
| `Stop` | `stop_hook_active` |

`tool_use_id` links the `PreToolUse` and `PostToolUse` events of one tool call. `context` exists but is reserved, and it is always `None` in Python.

**What a callback returns.** Returning `{}` changes nothing. Otherwise return an object with two kinds of fields:

- **Top-level fields:** `continue_`, `stopReason`, `systemMessage` (shown to the user), `reason` (feedback for Claude) and `suppressOutput`.
- **Event-specific fields** go inside `hookSpecificOutput`, which must include `hookEventName`.

| Event | Fields allowed inside `hookSpecificOutput` |
|---|---|
| `PreToolUse` | `permissionDecision` (`allow`, `deny`, `ask`, `defer`), `permissionDecisionReason`, `updatedInput`, `additionalContext` |
| `PostToolUse` | `additionalContext`, `updatedToolOutput` |
| `PostToolUseFailure` | `additionalContext` |
| `UserPromptSubmit` | `additionalContext` |

To block a call, use `permissionDecision: "deny"` inside `hookSpecificOutput`.

## Demo 2 (part 1): The event tracer

The tracer is a hook that does nothing except report. It prints three things: the event name from `input_data["hook_event_name"]`, the tool name if there is one, and the last six characters of `tool_use_id`.

Two details are worth noticing:

- `input_data.get("tool_name", "-")` is used because not every event has a tool. `UserPromptSubmit` and `Stop` carry no `tool_name`.
- `(tool_use_id or "-")` guards against `None`, because events that are not tied to a tool call have no id.

The hook returns `{}`, so it observes the run without changing it.

In [9]:
async def trace_event(input_data, tool_use_id, context):
    event = input_data["hook_event_name"]
    tool = input_data.get("tool_name", "-")
    short_id = (tool_use_id or "-")[-6:]
    print(f"[{event}] tool={tool} id={short_id}")
    return {}

## Demo 2 (part 2): Register one hook on four events

The same function, `trace_event`, is registered under four event names. There is **no matcher** on any of them, so each hook fires every time its event occurs. Only `Read` is allowed, and the prompt asks Claude to read the notes and count the action items.

**What to look for (approximate shape):** a `UserPromptSubmit` line, then a `PreToolUse` and a `PostToolUse` pair for `Read`, then a `Stop` line. The two `Read` lines should end with the **same short id**. That shared id is how you connect the before and the after of one tool call.

Do not expect every run to look identical, and do not assume any ordering beyond what you see on your screen.

In [10]:
async for message in query(
    prompt=(
        "Read /content/hooks_demo/drafts/meeting-notes.txt "
        "and tell me how many action items it lists."
    ),
    options=ClaudeAgentOptions(
        allowed_tools=["Read"],
        hooks={
            "UserPromptSubmit": [HookMatcher(hooks=[trace_event])],
            "PreToolUse": [HookMatcher(hooks=[trace_event])],
            "PostToolUse": [HookMatcher(hooks=[trace_event])],
            "Stop": [HookMatcher(hooks=[trace_event])],
        },
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if hasattr(message, "result"):
        print(message.result)

[UserPromptSubmit] tool=- id=1b72ad
[PreToolUse] tool=Read id=QwWgHn
[PostToolUse] tool=Read id=QwWgHn
[Stop] tool=- id=f0a951
The meeting notes file lists **3 action items**:

1. Asha sends the revised schedule by Friday
2. Ravi updates the changelog for version 1.2
3. Meera books the review room for Monday


## Matchers: choosing which calls a hook sees

A matcher is tested against the **tool name** and nothing else. These are the three forms:

| Form | Example | Matches |
|---|---|---|
| Exact | `"Bash"` | Only the `Bash` tool |
| Pipe list | `"Write\|Edit"` | `Write` or `Edit` |
| Regex | `"^mcp__"` | Any tool whose name starts with `mcp__` |
| *(omitted)* | `HookMatcher(hooks=[...])` | Every occurrence of the event |

A matcher never looks at file paths or arguments. If you want to treat `drafts/` differently from `published/`, the matcher cannot do it. The hook does it, by reading `tool_input`, exactly as Demo 1 did.

Demo 3 makes this visible by counting how often each matcher fires.

## Demo 3 (part 1): A factory for counting hooks

`fired` is a `Counter`, which works like a dictionary of counts. `make_counter(label)` is a small factory: each call builds a new hook that adds one to `fired[label]` and returns `{}`.

Building hooks this way lets us create three hooks that differ only in their label, which is exactly what the demo needs.

In [11]:
from collections import Counter

fired = Counter()

def make_counter(label):
    async def count_hook(input_data, tool_use_id, context):
        fired[label] += 1
        return {}
    return count_hook

## Demo 3 (part 2): Three matchers on one event

All three registrations are on `PreToolUse`:

| Label | Matcher | Expected |
|---|---|---|
| `FILE` | `"Write\|Edit"` | Counts file writes |
| `BASH` | `"Bash"` | Stays at zero, because Claude has no reason to run a shell command |
| `ALL` | *(none)* | Counts every tool call |

The prompt asks Claude to read the notes and then write a one-line summary. That gives us at least one `Read` and one `Write`. The hooks print nothing. The evidence is the counter, and the next cell prints it.

In [12]:
target = PROJECT / "drafts" / "summary.txt"
if target.exists():
    target.unlink()
fired.clear()

async for message in query(
    prompt=(
        "Read /content/hooks_demo/drafts/meeting-notes.txt, then create "
        "/content/hooks_demo/drafts/summary.txt containing a one-line summary of the notes."
    ),
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write"],
        hooks={
            "PreToolUse": [
                HookMatcher(matcher="Write|Edit", hooks=[make_counter("FILE")]),
                HookMatcher(matcher="Bash", hooks=[make_counter("BASH")]),
                HookMatcher(hooks=[make_counter("ALL")]),
            ],
        },
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if hasattr(message, "result"):
        print(message.result)

Done! I've created `/content/hooks_demo/drafts/summary.txt` with a one-line summary of the meeting notes. The summary captures the key details: the date of the weekly sync, the three attendees, and their respective action items.


## Demo 3 (part 3): Read the counter

**Expected (approximate):** `ALL` is at least 2, `FILE` is 1, and `BASH` is 0. `BASH` may be missing from the printout entirely, because a `Counter` only shows labels that were incremented. A missing label means zero.

`ALL` is larger than `FILE` because the no-matcher hook also saw the `Read` call. The last line confirms the file was really written.

In [13]:
print(dict(fired))
print("BASH count:", fired["BASH"])
print("summary.txt exists =", (PROJECT / "drafts" / "summary.txt").exists())

{'ALL': 2, 'FILE': 1}
BASH count: 0
summary.txt exists = True


## Execution rules: parallel hooks and decision priority

**Parallel execution.** All matching hooks for an event run **in parallel**, and completion order is not guaranteed. Write every hook to be independent. Never rely on another hook having run first.

**Decision priority.** When hooks return conflicting permission decisions, they resolve in this order:

`deny` > `defer` > `ask` > `allow`

One `deny` blocks the call, no matter how many hooks say `allow`.

**Two more rules about `allow`:**

- A hook that returns `allow` does **not** skip deny rules.
- It does mean that the `can_use_tool` callback is not called for that call. This is stated in the SDK source documentation.

**Catch your own exceptions.** An unhandled error inside a hook is a bug in your code, not in the agent. Wrap risky work in `try` and `except`, and return `{}` when something goes wrong, unless you intend to block.

```python
async def careful_hook(input_data, tool_use_id, context):
    try:
        ...  # your logic
    except Exception as error:
        print(f"Hook problem: {error}")
    return {}
```

## Demo 4 (part 1): Two hooks that disagree

`allow_hook` prints `[allow-hook ran]` and returns `permissionDecision: "allow"`. `deny_hook` prints `[deny-hook ran]` and returns `permissionDecision: "deny"` with a reason.

Each one prints its own name so that we can see **both** ran. Neither knows about the other, which is the independence the previous cell asked for.

In [14]:
async def allow_hook(input_data, tool_use_id, context):
    print("[allow-hook ran]")
    return {
        "hookSpecificOutput": {
            "hookEventName": input_data["hook_event_name"],
            "permissionDecision": "allow",
        }
    }


async def deny_hook(input_data, tool_use_id, context):
    print("[deny-hook ran]")
    return {
        "hookSpecificOutput": {
            "hookEventName": input_data["hook_event_name"],
            "permissionDecision": "deny",
            "permissionDecisionReason": "This demo denies every Write call.",
        }
    }

## Demo 4 (part 2): Register both and run

The two hooks are registered as **two separate `HookMatcher` entries**, both with `matcher="Write"` on `PreToolUse`. `Write` is in `allowed_tools`, so without a deny the file would be created.

**What to look for:** both `[allow-hook ran]` and `[deny-hook ran]` print, **in either order**. The order can differ between runs because the hooks run in parallel. If Claude tries the write again after the denial, you may see the pair more than once. That is fine.

In [15]:
target = PROJECT / "drafts" / "parallel-check.txt"
if target.exists():
    target.unlink()

async for message in query(
    prompt=(
        "Create /content/hooks_demo/drafts/parallel-check.txt "
        "containing the single line: hooks run in parallel."
    ),
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write"],
        hooks={
            "PreToolUse": [
                HookMatcher(matcher="Write", hooks=[allow_hook]),
                HookMatcher(matcher="Write", hooks=[deny_hook]),
            ],
        },
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if hasattr(message, "result"):
        print(message.result)

[allow-hook ran]
[deny-hook ran]
The Write call was blocked by a hook—specifically, a demo hook that denies every Write call. This is actually demonstrating the hooks system in action. The hook intercepted the tool call before it could execute.

If you'd like to create this file, you'd need to either:
1. Remove or modify the hook that's blocking Write calls
2. Use the Bash tool as an alternative to create the file

Would you like me to try creating the file using Bash instead?


## Demo 4 (part 3): Ground truth

**Expected:** `exists = False`. One `deny` outweighed one `allow`, so the write never happened.

In [16]:
path = PROJECT / "drafts" / "parallel-check.txt"
print(f"drafts/parallel-check.txt exists = {path.exists()}")

drafts/parallel-check.txt exists = False


## Reference: async output (read only, do not run)

Some hooks only perform a side effect, such as writing a log line or sending a notification. Such a hook can return an **async output** so that the agent continues without waiting for it:

```python
async def log_and_move_on(input_data, tool_use_id, context):
    # Hand the slow side effect to your own code here, then return at once.
    return {"async_": True, "asyncTimeout": 30000}
```

| Field | Meaning |
|---|---|
| `async_` | Set to `True` so the agent proceeds without waiting. The trailing underscore avoids the Python keyword. |
| `asyncTimeout` | Optional timeout in milliseconds for the background work |

An async output **cannot block, modify, or add context**, because the agent has already moved on. Use it only for side effects. We show it here as a reference and do not run it.

## Reference: timeouts

Each `HookMatcher` can set `timeout` in seconds. When you do not set one, the default is **600 seconds for most events** and **30 seconds for `UserPromptSubmit`**.

What happens when a hook times out depends on the event:

| Event | Result of a timeout |
|---|---|
| `PreToolUse` | The tool does not run, and Claude is told the hook did not respond |
| `PostToolUse` | The tool result is kept |
| `UserPromptSubmit` | The prompt is blocked |

Choose a timeout that fits the work your hook does, and keep hooks fast.

## Pitfalls checklist

- [ ] **Event names are case sensitive.** `PreToolUse` works. `preToolUse` does not.
- [ ] **A matcher matches tool names only.** Check paths and arguments inside the hook, using `tool_input`.
- [ ] **`updatedInput` must sit inside `hookSpecificOutput`.** At the top level it is ignored.
- [ ] **Always include `hookEventName`** inside `hookSpecificOutput`.
- [ ] **Hooks may not fire if the run hits `max_turns`,** because the session ends before the hooks can execute.
- [ ] **Prove a block with a tool that would otherwise have been allowed.** Put the tool in `allowed_tools`, and then check the filesystem.
- [ ] **Write each hook to be independent.** Parallel hooks finish in any order.

## Summary

In this notebook you:

1. Recalled the `PreToolUse` hook, which is **one of ten** Python hook events.
2. Learned the anatomy of a registration: event names as keys, lists of `HookMatcher` as values, and matchers that match **tool names only**.
3. Learned the contract: what `input_data` contains, and what a callback can return in the top-level fields and in `hookSpecificOutput`.
4. Saw that **all matching hooks run in parallel**, and that **deny wins** over every other decision.
5. Proved each idea with a demo: a protected folder, an event trace, a matcher counter, and a deny that beat an allow.

**Next:** Lecture 5.2 uses `PostToolUse` to build an audit log of every tool call.